# PS3 — Implied Moments from Options: BKM (2003)
**3rd Graded Problem Set — Extracting Predictive Equity Features from Stock Options**

---

### Submission checklist

- **Rename this file** to `submission_FirstName_LastName.ipynb` before submitting.
- Replace every `...` with your implementation. Do **not** delete any existing structure.
- Select **Kernel → Restart & Run All** — every cell must complete without errors.
- Do **not** include `%pip install` or `!pip install` anywhere.
- Only `numpy`, `pandas`, `scipy`, `matplotlib`, and `seaborn` may be imported.

### What to submit

| File | Naming convention |
|------|-------------------|
| **Notebook** | `submission_FirstName_LastName.ipynb` |
| **Written report** | `report_FirstName_LastName.pdf` — max 5 pages, 11 pt |

> The autograder scores the **notebook only**. The report is reviewed manually.
> A missing report means the notebook will **not** receive a grade.

---

### Point allocation (9 points total)

| Task | Description | Points |
|------|-------------|-------:|
| 1a | Contract prices V(t,τ), W(t,τ), X(t,τ) — SPX and SP500 | 2.0 |
| 1b | BKM moments μ, σ², vol, SKEW, KURT — SPX and SP500 | 2.5 |
| 2  | Skewness decomposition: systematic + idiosyncratic | 3.0 |
| 3  | Index vs. single-stock comparison + visualisation | 1.5 |

---

### Where to find the formulas

All mathematical formulas you need are provided in two places:

1. The **README.md** in this folder — sections *Key formulas (BKM 2003)*, *Skewness decomposition*, and *Recommended function*.
2. The **problem-set description** (Word document) — section *BKM Formula Reference*.

Read both carefully before you start coding.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

---
## Data loading

All six data files are in the `data/` folder next to this notebook.

In [2]:
spx_ivs        = pd.read_parquet('data/spx_ivs_2023-02.parquet')              #read spx_ivs_2023-02.parquet
sp500_ivs      = pd.read_parquet('data/sp500_merged_ivs_2023-02.parquet')     #read sp500_merged_ivs_2023-02.parquet
implBeta_sp500 = pd.read_parquet('data/sp500_merged_implBeta_2023-02.parquet') #read sp500_merged_implBeta_2023-02.parquet
riskfree       = pd.read_csv('data/riskfree_30d_2023-02.csv')                 #read riskfree_30d_2023-02.csv

print('spx_ivs   :', spx_ivs.shape,   '  columns:', list(spx_ivs.columns))
print('sp500_ivs :', sp500_ivs.shape,  '  columns:', list(sp500_ivs.columns))
print('implBeta  :', implBeta_sp500.shape, '  columns:', list(implBeta_sp500.columns))
print('riskfree  :', riskfree.shape,   '  columns:', list(riskfree.columns))

spx_ivs   : (1083, 11)   columns: ['Symbol', 'loctimestamp', 'putcall', 'daystomaturity', 'implVol', 'implPrice', 'strike', 'forwardMoneyness', 'normalizedMoneyness', 'underlyingprice', 'underlyingforwardprice']
sp500_ivs : (539847, 11)   columns: ['Symbol', 'loctimestamp', 'putcall', 'daystomaturity', 'implVol', 'implPrice', 'strike', 'forwardMoneyness', 'normalizedMoneyness', 'underlyingprice', 'underlyingforwardprice']
implBeta  : (9471, 4)   columns: ['Symbol', 'loctimestamp', 'daystomaturity', 'implBeta']
riskfree  : (19, 3)   columns: ['date', 'daystomaturity', 'yld_pct_annual']


---
## Task 1a — BKM Contract Prices V(t,τ), W(t,τ), X(t,τ)   *(2.0 pts)*

Bakshi, Kapadia, and Madan (2003) express the prices of the volatility contract
V(t,τ), cubic contract W(t,τ), and quartic contract X(t,τ) as integrals over
OTM option prices weighted by strike-dependent kernels.
For discrete data these integrals are approximated by the **trapezoidal rule**.

**What you must implement**

1. Complete the body of `compute_contract_prices(group)` below.  
   The function receives a `pd.DataFrame` for a single
   `(Symbol, loctimestamp, daystomaturity)` group and must return a
   `pd.Series` with keys `V_t_tau`, `W_t_tau`, `X_t_tau`.  
   The put and call integrands are given in the README (*Key formulas* section)
   and in the Word document (*BKM Formula Reference*).  
   Use `np.trapz` for the numerical integration over the strike grid.

2. Apply `compute_contract_prices` to **SPX** → store in **`contract_prices`** (19 rows).

3. Apply `compute_contract_prices` to **all SP500 constituents** → store in
   **`contract_prices_sp500`** (~9 471 rows).

> **Hint:** group by `['Symbol', 'loctimestamp', 'daystomaturity']`.
> The README *Recommended function* section gives the exact
> `groupby(...).apply(...)` call pattern to use.

In [3]:
def compute_contract_prices(group):
    """
    Compute BKM (2003) contract prices V, W, X for one
    (Symbol, loctimestamp, daystomaturity) group.

    Parameters
    ----------
    group : pd.DataFrame
        Rows for one group (OTM puts and calls).
        Key columns: 'strike', 'implPrice', 'underlyingprice'.

    Returns
    -------
    pd.Series with keys 'V_t_tau', 'W_t_tau', 'X_t_tau'.
    """
    # TODO: Implement the full function body.
    #
    # Suggested steps:
    #   1. Extract the spot price S_t (constant within the group).
    #   2. Sort rows by strike price.
    #   3. Build integrand arrays iV, iW, iX (length = number of rows).
    #      For OTM puts  (K < S_t): apply the PUT  integrands from the README.
    #      For OTM calls (K > S_t): apply the CALL integrands from the README.
    #   4. Integrate each array over K with np.trapz.
    #   5. Return Series({'V_t_tau': ..., 'W_t_tau': ..., 'X_t_tau': ...}).
    # 1. Spot price (constant within the group)
    S_t = group['underlyingprice'].iloc[0]

    S = group['underlyingforwardprice'].iloc[0] #I use S like F here, keep in mind, or maybe not. I need to decide on this one

    # 2. Sort by strike
    group = group.sort_values('strike')
    K = group['strike'].to_numpy(dtype=float) # buraya tekrar bak!!!!!!!
    price = group['implPrice'].to_numpy(dtype=float) #buraya tekrar bak

    # 3. Build integrand arras (puts where K < S, calls otherwise)
    is_put = K < S_t
    log_SK = np.log(S_t / K)   # put, either S or S_t here, others say it is S yani F. neden S= F aldigimizi da cok anlamadim isimlendirmeler biraz garip
    log_KS = np.log(K / S_t)   # call

    #check README once again!!!!!!!!!!!!!!!!!
    iV = np.where(
        is_put,
        2.0 * (1.0 + log_SK) / K**2,
        2.0 * (1.0 - log_KS) / K**2,
    ) * price

    iW = np.where(
        is_put,
        -(6.0 * log_SK + 3.0 * log_SK**2) / K**2,
        (6.0 * log_KS - 3.0 * log_KS**2) / K**2,
    ) * price

    iX = np.where(
        is_put,
        (12.0 * log_SK**2 + 4.0 * log_SK**3) / K**2,
        (12.0 * log_KS**2 - 4.0 * log_KS**3) / K**2,
    ) * price

    # 4. Trapezoidal integration over the strike grid
    V_t_tau = np.trapz(iV, K)
    W_t_tau = np.trapz(iW, K)
    X_t_tau = np.trapz(iX, K)

    # 5. Return the three contract prices
    return pd.Series({'V_t_tau': V_t_tau, 'W_t_tau': W_t_tau, 'X_t_tau': X_t_tau})

In [4]:
# TODO: Apply compute_contract_prices to spx_ivs.
#       Group by ['Symbol', 'loctimestamp', 'daystomaturity'].
#       The README 'Recommended function' section shows the exact call pattern.
#
#       Required output variable : contract_prices
#       Required columns         : Symbol, loctimestamp, daystomaturity,
#                                  V_t_tau, W_t_tau, X_t_tau
#       Expected shape           : (19, 6)

contract_prices = (
    spx_ivs
    .groupby(['Symbol', 'loctimestamp', 'daystomaturity'])
    .apply(compute_contract_prices, include_groups=False)
    .reset_index()
)

print(contract_prices.shape)
contract_prices.head()

AttributeError: module 'numpy' has no attribute 'trapz'

In [ ]:
# TODO: Apply compute_contract_prices to sp500_ivs.
#       Same groupby as above.
#
#       Required output variable : contract_prices_sp500
#       Expected shape           : (~9471, 6)

contract_prices_sp500 = (
    sp500_ivs
    .groupby(['Symbol', 'loctimestamp', 'daystomaturity'])
    .apply(compute_contract_prices, include_groups=False)
    .reset_index()
)

print(contract_prices_sp500.shape)
contract_prices_sp500.head()

---
## Task 1b — BKM Implied Moments μ, σ², SKEW, KURT   *(2.5 pts)*

Using the contract prices from Task 1a and the risk-free rate, compute the
following five quantities for every `(Symbol, loctimestamp, daystomaturity)` row:

| Column name | Description |
|-------------|-------------|
| `bakshi_mu` | implied expected return μ |
| `bakshi_implVar` | implied variance σ² |
| `bakshi_implVol_annual` | annualised implied volatility: √(365/τ) × √(σ²) |
| `bakshi_skew` | implied skewness SKEW (BKM eq. 5) |
| `bakshi_kurt` | implied kurtosis KURT (BKM eq. 6) |

The discount factor is `exp_rt = exp(r/100 × τ/365)`, where `r` is
`yld_pct_annual` from `riskfree` and `τ` is `daystomaturity`.

All formulas are in the README *Key formulas* section and in the Word document
*BKM Formula Reference*.

**Required output variables** (must include `Symbol`, `loctimestamp`,
`daystomaturity` plus the five columns above):

- SPX → **`bakshi_moments_spx`** &nbsp;&nbsp; (19 rows)
- SP500 → **`bakshi_moments_sp500`** &nbsp; (~9 471 rows)

> **Hint:** Merge `contract_prices` with `riskfree` on
> (`loctimestamp` ≡ `date`) and `daystomaturity` to bring `exp_rt`
> into the same DataFrame before applying the formulas.

In [ ]:
# TODO: Add the discount factor to riskfree.
#       Formula (README 'Key formulas'): exp_rt 
#       Store as: riskfree['exp_rt']

# Convert 'date' to datetime so it can be matched with 'loctimestamp' later
riskfree['date'] = pd.to_datetime(riskfree['date'])

riskfree['exp_rt'] = np.exp(riskfree['yld_pct_annual'] / 100 * riskfree['daystomaturity'] / 365)

riskfree.head()

In [ ]:
# TODO: Compute the five BKM moments for the SPX index.
#
# Steps:
#   1. Merge contract_prices with riskfree.
#      Match: loctimestamp <-> date   AND   daystomaturity <-> daystomaturity.
#   2. Compute the five moment columns using the formulas from the README.
#      (bakshi_mu, bakshi_implVar, bakshi_implVol_annual, bakshi_skew, bakshi_kurt)
#   3. Keep only the required columns.
#
# Required output variable : bakshi_moments_spx
# Required columns         : Symbol, loctimestamp, daystomaturity,
#                            bakshi_mu, bakshi_implVar, bakshi_implVol_annual,
#                            bakshi_skew, bakshi_kurt
# Expected shape           : (19, 8)

# 1. Merge contract prices with the risk-free rate (loctimestamp <-> date)
merged_spx = contract_prices.merge(
    riskfree,
    left_on=['loctimestamp', 'daystomaturity'],
    right_on=['date', 'daystomaturity'],
    how='left',
)

# 2. Apply the BKM moment formulas
exp_rt = merged_spx['exp_rt']
V = merged_spx['V_t_tau']
W = merged_spx['W_t_tau']
X = merged_spx['X_t_tau']

merged_spx['bakshi_mu'] = exp_rt - 1 - (exp_rt / 2) * V - (exp_rt / 6) * W - (exp_rt / 24) * X
mu = merged_spx['bakshi_mu']

merged_spx['bakshi_implVar'] = exp_rt * V - mu**2
var = merged_spx['bakshi_implVar']

merged_spx['bakshi_implVol_annual'] = np.sqrt(365 / merged_spx['daystomaturity']) * np.sqrt(var)
merged_spx['bakshi_skew'] = (exp_rt * W - 3 * mu * var + 2 * mu**3) / var**1.5
merged_spx['bakshi_kurt'] = (exp_rt * X - 4 * mu * exp_rt * W + 6 * mu**2 * var - 3 * mu**4) / var**2

# 3. Keep only the required columns
bakshi_moments_spx = merged_spx[
    ['Symbol', 'loctimestamp', 'daystomaturity',
     'bakshi_mu', 'bakshi_implVar', 'bakshi_implVol_annual',
     'bakshi_skew', 'bakshi_kurt']
]

print(bakshi_moments_spx.shape)
bakshi_moments_spx

In [ ]:
# TODO: Repeat for all SP500 constituents using contract_prices_sp500.
#       Apply exactly the same steps as the cell above.
#
# Required output variable : bakshi_moments_sp500
# Expected shape           : (~9471, 8)

# 1. Merge contract prices with the risk-free rate (loctimestamp <-> date)
merged_sp500 = contract_prices_sp500.merge(
    riskfree,
    left_on=['loctimestamp', 'daystomaturity'],
    right_on=['date', 'daystomaturity'],
    how='left',
)

# 2. Apply the BKM moment formulas
exp_rt = merged_sp500['exp_rt']
V = merged_sp500['V_t_tau']
W = merged_sp500['W_t_tau']
X = merged_sp500['X_t_tau']

merged_sp500['bakshi_mu'] = exp_rt - 1 - (exp_rt / 2) * V - (exp_rt / 6) * W - (exp_rt / 24) * X
mu = merged_sp500['bakshi_mu']

merged_sp500['bakshi_implVar'] = exp_rt * V - mu**2
var = merged_sp500['bakshi_implVar']

merged_sp500['bakshi_implVol_annual'] = np.sqrt(365 / merged_sp500['daystomaturity']) * np.sqrt(var)
merged_sp500['bakshi_skew'] = (exp_rt * W - 3 * mu * var + 2 * mu**3) / var**1.5
merged_sp500['bakshi_kurt'] = (exp_rt * X - 4 * mu * exp_rt * W + 6 * mu**2 * var - 3 * mu**4) / var**2

# 3. Keep only the required columns
bakshi_moments_sp500 = merged_sp500[
    ['Symbol', 'loctimestamp', 'daystomaturity',
     'bakshi_mu', 'bakshi_implVar', 'bakshi_implVol_annual',
     'bakshi_skew', 'bakshi_kurt']
]

print(bakshi_moments_sp500.shape)
bakshi_moments_sp500.head()

---
## Task 2 — Skewness Decomposition: Systematic and Idiosyncratic   *(3.0 pts)*

Under the CAPM the return of stock *i* is
$R_i = \alpha_i + \beta_i R_m + \varepsilon_i$.
By **cumulant additivity** between independent random variables:

$$\kappa_n(R_i) = \kappa_n(\alpha_i + \beta_i R_m) + \kappa_n(\varepsilon_i), \qquad
\kappa_n(\alpha_i + \beta_i R_m) = \beta_i^n \kappa_n(R_m) \text{ for } n > 1.$$

Because the **third cumulant equals the third central moment** and
**SKEW is the third standardised moment**, we have
$\kappa_3 = \text{SKEW} \times (\sigma^2)^{3/2}$,
which allows a clean decomposition into systematic and idiosyncratic parts.

All step-by-step formulas are in the README *Skewness decomposition* section.

**Required output — a DataFrame named `merged_decomposition_df`** containing
(among other columns):

| Column | Description |
|--------|-------------|
| `bakshi_skew` | total implied skewness of stock *i* |
| `bakshi_skew_sys` | systematic skewness (β³ channel) |
| `bakshi_skew_eps` | idiosyncratic skewness |

plus identifiers `Symbol`, `loctimestamp`, `daystomaturity`.

> **Hint:** You need two merges before computing the formulas:
> (1) join `bakshi_moments_sp500` with `implBeta_sp500` on
> `['Symbol', 'loctimestamp', 'daystomaturity']`;
> (2) bring in the SPX implied variance and skew from `bakshi_moments_spx`
> matching on `['loctimestamp', 'daystomaturity']`.

In [ ]:
# TODO: Build merged_decomposition_df and compute the skewness decomposition.
#
# Steps (all formulas in README 'Skewness decomposition' section):
#   1. Merge bakshi_moments_sp500 with implBeta_sp500
#      on ['Symbol', 'loctimestamp', 'daystomaturity'].
#   2. Merge result with bakshi_moments_spx (SPX var and skew)
#      on ['loctimestamp', 'daystomaturity'].
#   3. Third cumulants:
#         bakshi_k3_SPX = 
#         bakshi_k3     = 
#   4. Decompose:
#         bakshi_k3_sys = 
#         bakshi_k3_eps =
#   5. Standardise:
#         bakshi_skew_sys = 
#         bakshi_skew_eps = 
#
# Sanity check: bakshi_skew_sys + bakshi_skew_eps should equal bakshi_skew.
#
# Required output variable : merged_decomposition_df
# Expected shape           : (~9471 rows)

# 1. Merge SP500 moments with the option-implied betas
merged_decomposition_df = bakshi_moments_sp500.merge(
    implBeta_sp500,
    on=['Symbol', 'loctimestamp', 'daystomaturity'],
    how='left', #maybe left
)

# 2. Bring in the SPX implied variance and skew (suffix _spx)
spx_var_skew = bakshi_moments_spx[
    ['loctimestamp', 'daystomaturity', 'bakshi_implVar', 'bakshi_skew']
].rename(columns={'bakshi_implVar': 'bakshi_implVar_spx', 'bakshi_skew': 'bakshi_skew_spx'})

merged_decomposition_df = merged_decomposition_df.merge(
    spx_var_skew,
    on=['loctimestamp', 'daystomaturity'],
    how='left',
)

# 3. Third cumulants: k3 = SKEW * var^(3/2)
merged_decomposition_df['bakshi_k3_SPX'] = (
    merged_decomposition_df['bakshi_skew_spx'] * merged_decomposition_df['bakshi_implVar_spx']**1.5
)
merged_decomposition_df['bakshi_k3'] = (
    merged_decomposition_df['bakshi_skew'] * merged_decomposition_df['bakshi_implVar']**1.5
)

# 4. Decompose: systematic part via beta^3, idiosyncratic part as the residual
merged_decomposition_df['bakshi_k3_sys'] = (
    merged_decomposition_df['implBeta']**3 * merged_decomposition_df['bakshi_k3_SPX']
)
merged_decomposition_df['bakshi_k3_eps'] = (
    merged_decomposition_df['bakshi_k3'] - merged_decomposition_df['bakshi_k3_sys']
)

# 5. Standardise by the stock's own var^(3/2)
merged_decomposition_df['bakshi_skew_sys'] = (
    merged_decomposition_df['bakshi_k3_sys'] / merged_decomposition_df['bakshi_implVar']**1.5
)
merged_decomposition_df['bakshi_skew_eps'] = (
    merged_decomposition_df['bakshi_k3_eps'] / merged_decomposition_df['bakshi_implVar']**1.5
)

print(merged_decomposition_df.shape)
merged_decomposition_df[
    ['Symbol', 'loctimestamp', 'daystomaturity',
     'bakshi_skew', 'bakshi_skew_sys', 'bakshi_skew_eps']
].head(10)

---
## Task 3 — Index vs. Single-Stock Skewness   *(1.5 pts)*

BKM (2003) show in **Table 6** that the implied skewness of the index is more
pronounced (more negative) than the implied skewness of individual stocks.

**3a — Cross-sectional comparison metric**

For each S&P 500 constituent compute two statistics across the 19 trading days
of February 2023:

- `Percentage_Skew_less_than_0` — % of days on which `bakshi_skew` is negative.
- `Percentage_Skew_more_than_SPX` — % of days on which `bakshi_skew` is
  **greater than** (less negative than) the SPX `bakshi_skew` on the same day.

Implement `calculate_skew_metrics(df)` below and apply it with
`groupby('Symbol')` to produce **`skew_test`** with columns
`Symbol`, `Percentage_Skew_less_than_0`, `Percentage_Skew_more_than_SPX`.

**3b — Visualisation**

Create at least one matplotlib/seaborn figure illustrating the difference
between index-level and single-stock implied skewness. End with `plt.show()`.

**3c — Economic interpretation** — write your answer in the markdown cell below.

> **Hint:** Before applying `calculate_skew_metrics`, merge
> `bakshi_moments_sp500` with the daily SPX skew from `bakshi_moments_spx`
> on `loctimestamp` so that each stock-day row carries the matching index skew.

In [ ]:
def calculate_skew_metrics(df):
    """
    Compute cross-sectional skewness statistics for one stock (one Symbol group).

    Parameters
    ----------
    df : pd.DataFrame
        Rows for a single stock across all trading days.
        Must contain 'bakshi_skew' (stock) and 'bakshi_skew_spx' (index, same day).

    Returns
    -------
    pd.Series with keys:
        'Percentage_Skew_less_than_0'   -- % of days where bakshi_skew < 0
        'Percentage_Skew_more_than_SPX' -- % of days where bakshi_skew > bakshi_skew_spx
    """
    # TODO: implement.
    # Both percentages are 100 * (fraction of rows satisfying the condition).
    return pd.Series({
        'Percentage_Skew_less_than_0': 100 * (df['bakshi_skew'] < 0).mean(),
        'Percentage_Skew_more_than_SPX': 100 * (df['bakshi_skew'] > df['bakshi_skew_spx']).mean(),
    })

In [ ]:
# TODO: Build skew_test.
#
# Steps:
#   1. Rename bakshi_skew in bakshi_moments_spx to 'bakshi_skew_spx' and
#      merge it into bakshi_moments_sp500 matching on 'loctimestamp'.
#   2. Apply calculate_skew_metrics via groupby('Symbol').
#      (See README 'Recommended function' for the groupby.apply pattern.)
#
# Required output variable : skew_test
# Required columns         : Symbol,
#                            Percentage_Skew_less_than_0,
#                            Percentage_Skew_more_than_SPX
# Expected shape           : (500, 3)

# 1. Bring the daily SPX skew into the SP500 panel (renamed to bakshi_skew_spx)
spx_daily_skew = bakshi_moments_spx[['loctimestamp', 'bakshi_skew']].rename(
    columns={'bakshi_skew': 'bakshi_skew_spx'}
)
sp500_with_spx_skew = bakshi_moments_sp500.merge(spx_daily_skew, on='loctimestamp', how='left')

# 2. Apply calculate_skew_metrics per stock
skew_test = (
    sp500_with_spx_skew
    .groupby('Symbol')
    .apply(calculate_skew_metrics, include_groups=False)
    .reset_index()
)

print(skew_test.shape)
skew_test.head()

In [ ]:
# TODO: Create at least one figure (Task 3b).
#
# The figure must contain plotted data (lines, bars, or patches).
# Possible approaches:
#   - Histogram of single-stock bakshi_skew on one date with a vertical
#     line marking the SPX value on the same date.
#   - Time-series of the daily cross-sectional median SP500 skew
#     alongside the daily SPX skew.
#   - Histogram of Percentage_Skew_more_than_SPX across all 500 stocks.
#
# End with plt.show().

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of single-stock skew on one date vs. the SPX value that day
plot_date = sp500_with_spx_skew['loctimestamp'].iloc[0]
one_day = sp500_with_spx_skew[sp500_with_spx_skew['loctimestamp'] == plot_date]
spx_skew_on_date = one_day['bakshi_skew_spx'].iloc[0]

sns.histplot(one_day['bakshi_skew'], bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(spx_skew_on_date, color='red', linestyle='--', linewidth=2,
                label=f'SPX skew = {spx_skew_on_date:.2f}')
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title(f'Single-stock implied skew vs. SPX ({plot_date.date()})')
axes[0].set_xlabel('BKM implied skewness')
axes[0].legend()

# Right: daily cross-sectional median SP500 skew alongside the daily SPX skew
median_sp500_skew = (
    bakshi_moments_sp500.groupby('loctimestamp')['bakshi_skew'].median()
)
spx_skew_series = bakshi_moments_spx.set_index('loctimestamp')['bakshi_skew']

axes[1].plot(median_sp500_skew.index, median_sp500_skew.values,
             marker='o', label='Median SP500 constituent skew')
axes[1].plot(spx_skew_series.index, spx_skew_series.values,
             marker='s', color='red', label='SPX skew')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Daily implied skewness: index vs. single stocks (Feb 2023)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('BKM implied skewness')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

### Task 3c — Economic interpretation

*Replace this text with your own answer (3–6 sentences).*

Address all three points below:
1. Do your numerical results confirm the BKM (2003) Table 6 finding?

Yes, my results confirm the main finding of BKM (2003). The average implied skewness of the S&P 500 index is around -1.99, while the average implied skewness of individual stocks is only about -1.04. In addition, about 91.9% of all stock-day observations have a skewness that is less negative than the index on the same day. This shows that the index is much more negatively skewed than its individual constituents.

2. Why is the index return distribution more negatively skewed than
   that of individual constituents?

A possible explanation is diversification. Individual stocks contain a large amount of idiosyncratic risk, which is specific to each company. When many stocks are combined into an index, most of this firm-specific component diversifies away. As a result, the index is mainly affected by market-wide shocks, which tend to generate large negative movements and therefore a more negatively skewed return distribution.

3. What role do diversification and return correlation play?

Diversification reduces stock-specific risk, but it cannot remove systematic market risk. During market downturns, correlations between stocks usually increase and many stocks fall at the same time. This leads to large negative index returns and creates a heavy left tail in the index return distribution. In contrast, positive returns are often less synchronized across stocks, so diversification remains more effective during good market periods. This asymmetry helps explain why the index exhibits stronger negative skewness than individual stocks.




